# DDPM Task 2 — Analysis

Uses the step-stamped checkpoints synced from all 5 vast.ai runs (`gdrive:lab1-ckpts/<label>/`) to build:

1. Table — final FID per config
2. Figure — scheduler trajectory comparison (linear/quad/cosine, noise predictor)
3. Figure — predictor sample comparison (noise/x0/mean, linear scheduler, 8 samples each)
4. Figure + Table — FID vs. training step, all configs, plus convergence-speed table
5. Figure — sample quality progression over training (linear+noise)
6. Figure — training loss curves overlaid, all configs

Run cells top to bottom. Requires the `lab1-ckpts` folder to already be in your Google Drive (from the real-time sync during training).

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/daiyunwei1998/lab1.git
%cd lab1/image_diffusion_todo
!pip install -q dotmap pytorch_lightning jaxtyping

In [ ]:
# Downloads AFHQ and builds data/afhq/eval/, the reference set FID is measured against.
# Needed fresh here -- the dataset itself was never synced to Drive, only checkpoints.
!python dataset.py

In [ ]:
import torch, os, glob, re, subprocess
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
from model import DiffusionModule
from network import UNet
from scheduler import DDPMScheduler
from dataset import tensor_to_pil_image

device = "cuda" if torch.cuda.is_available() else "cpu"
DRIVE_ROOT = "/content/drive/MyDrive/lab1-ckpts"

# label -> (beta scheduler mode, predictor parameterization)
CONFIGS = {
    "linear_noise": {"mode": "linear", "predictor": "noise"},
    "quad_noise":   {"mode": "quad",   "predictor": "noise"},
    "cosine_noise": {"mode": "cosine", "predictor": "noise"},
    "linear_x0":    {"mode": "linear", "predictor": "x0"},
    "linear_mean":  {"mode": "linear", "predictor": "mean"},
}

def ckpt_dir(label):
    cfg = CONFIGS[label]
    return f"{DRIVE_ROOT}/{label}/predictor_{cfg['predictor']}/beta_{cfg['mode']}"

def load_checkpoint(path):
    """hparams in the .ckpt already carry the fully-built network + scheduler objects
    (see model.py's save/load), so this reconstructs a DiffusionModule directly from them
    rather than needing to know the original UNet constructor args."""
    dic = torch.load(path, map_location=device, weights_only=False)
    network = dic["hparams"]["network"].to(device)
    var_scheduler = dic["hparams"]["var_scheduler"].to(device)
    predictor = dic["hparams"].get("predictor", "noise")
    ddpm = DiffusionModule(network, var_scheduler, predictor=predictor).to(device)
    ddpm.load_state_dict(dic["state_dict"])
    ddpm.eval()
    return ddpm

def generate_samples(ddpm, n, out_dir, batch_size=64):
    os.makedirs(out_dir, exist_ok=True)
    idx, remaining = 0, n
    with torch.no_grad():
        while remaining > 0:
            b = min(batch_size, remaining)
            samples = ddpm.sample(b)
            for im in tensor_to_pil_image(samples):
                im.save(f"{out_dir}/{idx}.png")
                idx += 1
            remaining -= b
    return out_dir

def compute_fid(gen_dir, eval_dir="data/afhq/eval"):
    result = subprocess.run(["python", "fid/measure_fid.py", eval_dir, gen_dir],
                             capture_output=True, text=True)
    line = [l for l in result.stdout.splitlines() if l.startswith("FID:")][-1]
    return float(line.split("FID:")[1].strip())

def all_ckpt_steps(label):
    d = ckpt_dir(label)
    files = glob.glob(f"{d}/step=*.ckpt")
    return sorted(int(re.search(r"step=(\d+)\.ckpt", f).group(1)) for f in files)

def latest_traj(label):
    d = ckpt_dir(label)
    files = glob.glob(f"{d}/step=*-traj.png")
    steps = [int(re.search(r"step=(\d+)-traj", f).group(1)) for f in files]
    best = max(steps)
    return f"{d}/step={best}-traj.png", best

print("Configs found:", {l: ckpt_dir(l) for l in CONFIGS})

## 1. Table — final FID per config

The graded "best FID" number, computed fresh from each config's final (`last.ckpt`) weights -- also doubles as the scheduler/predictor comparison table.

In [ ]:
N_SAMPLES_FINAL = 2048  # match the assignment's own FID sample count

rows = []
for label, cfg in CONFIGS.items():
    d = ckpt_dir(label)
    ts = torch.load(f"{d}/train_state.pt", map_location="cpu", weights_only=False)
    step = ts["step"]

    ddpm = load_checkpoint(f"{d}/last.ckpt")
    gen_dir = f"/content/gen/{label}_final"
    generate_samples(ddpm, N_SAMPLES_FINAL, gen_dir)
    fid = compute_fid(gen_dir)
    rows.append({"label": label, "scheduler": cfg["mode"], "predictor": cfg["predictor"],
                 "final_step": step, "FID": fid})
    print(f"{label}: step={step} FID={fid:.4f}")

    del ddpm
    torch.cuda.empty_cache()

df_final = pd.DataFrame(rows)
df_final

## 2. Figure — scheduler trajectory comparison

`linear` vs `quad` vs `cosine` (all noise predictor). Reuses the trajectory strips already rendered during training -- note these are from the *last periodic checkpoint before* the final step (e.g. step 98000 not 100000), since trajectory images are only written inside the periodic log_interval block, which never fires exactly at the final step.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(14, 12))
for ax, label in zip(axs, ["linear_noise", "quad_noise", "cosine_noise"]):
    path, step = latest_traj(label)
    ax.imshow(Image.open(path))
    ax.axis("off")
    ax.set_title(f"{CONFIGS[label]['mode']} scheduler (trajectory at step {step})", fontsize=14)
plt.tight_layout()
plt.savefig("fig1_scheduler_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Figure — predictor sample comparison

`noise` vs `x0` vs `mean` (all linear scheduler), 8 fresh samples each from the final checkpoint (regenerated here, not reusing training-time snapshots, so all three rows are directly comparable).

In [ ]:
predictor_labels = ["linear_noise", "linear_x0", "linear_mean"]
fig, axs = plt.subplots(3, 8, figsize=(24, 9.5))

for row, label in enumerate(predictor_labels):
    d = ckpt_dir(label)
    ddpm = load_checkpoint(f"{d}/last.ckpt")
    with torch.no_grad():
        samples = ddpm.sample(8)
    pil_images = tensor_to_pil_image(samples)
    for col, im in enumerate(pil_images):
        axs[row, col].imshow(im)
        axs[row, col].axis("off")
    axs[row, 0].text(-0.3, 0.5, CONFIGS[label]["predictor"], fontsize=16, rotation=90,
                      va="center", ha="center", transform=axs[row, 0].transAxes)
    del ddpm
    torch.cuda.empty_cache()

plt.suptitle("Predictor comparison: 8 samples each (linear scheduler)", fontsize=16)
plt.tight_layout()
plt.savefig("fig2_predictor_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Figure + Table — FID vs. training step (all checkpoints, all configs)

The result only possible because every checkpoint was kept, not just the final one: shows *when* each config actually converges, and whether the final checkpoint is even the best one. Evaluates every `step=N.ckpt` found per config -- can be slow, this is the heaviest cell in the notebook.

In [ ]:
N_SAMPLES_CURVE = 500  # smaller than the final-FID count, purely for wall-clock reasons across many checkpoints

fid_curves = {}
for label in CONFIGS:
    d = ckpt_dir(label)
    steps = all_ckpt_steps(label)
    curve = []
    for step in steps:
        ddpm = load_checkpoint(f"{d}/step={step}.ckpt")
        gen_dir = f"/content/gen/{label}_{step}"
        generate_samples(ddpm, N_SAMPLES_CURVE, gen_dir)
        fid = compute_fid(gen_dir)
        curve.append((step, fid))
        print(f"{label}  step={step:>6}  FID={fid:.4f}")
        del ddpm
        torch.cuda.empty_cache()
    fid_curves[label] = curve

In [ ]:
plt.figure(figsize=(11, 6))
for label, curve in fid_curves.items():
    steps_, fids_ = zip(*curve)
    plt.plot(steps_, fids_, marker="o", label=label)
plt.xlabel("Training step")
plt.ylabel("FID (lower is better)")
plt.title("FID vs. training step, all 5 configs")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("fig3_fid_vs_step.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Convergence-speed table: earliest step each config's FID first dropped below a threshold.
conv_rows = []
for label, curve in fid_curves.items():
    first_below_20 = next((s for s, f in curve if f < 20), None)
    first_below_15 = next((s for s, f in curve if f < 15), None)
    best_step, best_fid = min(curve, key=lambda sf: sf[1])
    conv_rows.append({
        "label": label,
        "first_step_FID<20": first_below_20,
        "first_step_FID<15": first_below_15,
        "best_step": best_step,
        "best_FID": round(best_fid, 4),
    })
df_convergence = pd.DataFrame(conv_rows)
df_convergence

## 5. Figure — sample quality progression (linear + noise)

The longest, most-checkpointed run -- visual complement to the FID curve above, showing the same convergence directly in image space.

In [ ]:
label = "linear_noise"
d = ckpt_dir(label)
steps_avail = all_ckpt_steps(label)
chosen_steps = steps_avail[:: max(1, len(steps_avail) // 5)][:5]
if steps_avail[-1] not in chosen_steps:
    chosen_steps[-1] = steps_avail[-1]

fig, axs = plt.subplots(len(chosen_steps), 8, figsize=(24, 3 * len(chosen_steps)))
for row, step in enumerate(chosen_steps):
    ddpm = load_checkpoint(f"{d}/step={step}.ckpt")
    with torch.no_grad():
        samples = ddpm.sample(8)
    pil_images = tensor_to_pil_image(samples)
    for col, im in enumerate(pil_images):
        axs[row, col].imshow(im)
        axs[row, col].axis("off")
    axs[row, 0].text(-0.3, 0.5, f"step {step}", fontsize=14, rotation=90,
                      va="center", ha="center", transform=axs[row, 0].transAxes)
    del ddpm
    torch.cuda.empty_cache()

plt.suptitle("linear + noise: sample quality over training", fontsize=16)
plt.tight_layout()
plt.savefig("fig4_progression.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Figure — training loss curves overlaid

Pulled directly from each config's `train_state.pt` (`losses` list, saved throughout training), smoothed with a moving average since raw per-step MSE loss is noisy.

In [ ]:
plt.figure(figsize=(11, 6))
WINDOW = 200
for label in CONFIGS:
    d = ckpt_dir(label)
    ts = torch.load(f"{d}/train_state.pt", map_location="cpu", weights_only=False)
    losses = np.array(ts["losses"])
    if len(losses) >= WINDOW:
        smoothed = np.convolve(losses, np.ones(WINDOW) / WINDOW, mode="valid")
    else:
        smoothed = losses
    plt.plot(smoothed, label=label, alpha=0.85)
plt.xlabel("Training step")
plt.ylabel(f"Loss ({WINDOW}-step moving average)")
plt.title("Training loss curves, all 5 configs")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("fig5_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Save everything for the report

Copies the final table CSVs and all figure PNGs into Drive so they survive the Colab runtime shutting down.

In [ ]:
OUT_DIR = "/content/drive/MyDrive/lab1-analysis"
os.makedirs(OUT_DIR, exist_ok=True)

df_final.to_csv(f"{OUT_DIR}/table1_final_fid.csv", index=False)
df_convergence.to_csv(f"{OUT_DIR}/table2_convergence.csv", index=False)

import shutil
for f in glob.glob("fig*.png"):
    shutil.copy(f, OUT_DIR)

print("Saved to", OUT_DIR)
print(os.listdir(OUT_DIR))